# Row Echelon Form in General

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "Row Echelon Form in General" (DeepLearning.AI).*

The [2×2 procedure](./C1_W2_L2_row_echelon_form.ipynb) generalizes to any matrix. A matrix is in **row echelon form (REF)** when it forms a descending staircase of **pivots** (each row's leftmost non-zero entry):

1. **Zero rows sit at the bottom.**
2. **Every non-zero row has a pivot** (its leftmost non-zero entry).
3. **Each pivot is strictly to the right** of the pivot in the row above.

$$ \begin{bmatrix} \boxed{2} & * & * & * & * \\ 0 & \boxed{1} & * & * & * \\ 0 & 0 & \boxed{3} & * & * \\ 0 & 0 & 0 & \boxed{-5} & * \\ 0 & 0 & 0 & 0 & \boxed{1} \end{bmatrix} $$

In [1]:
import numpy as np
from fractions import Fraction as F

def to_ref(M, scale_pivots_to_one=True):
    """Row echelon form. Returns (REF matrix, list of pivot column indices)."""
    M = [[F(x) for x in row] for row in M]
    rows, cols = len(M), len(M[0])
    pivots, prow = [], 0
    for col in range(cols):
        piv = next((r for r in range(prow, rows) if M[r][col] != 0), None)
        if piv is None:
            continue                       # no pivot in this column -> staircase skips it
        M[prow], M[piv] = M[piv], M[prow]
        if scale_pivots_to_one:
            M[prow] = [x / M[prow][col] for x in M[prow]]
        for r in range(prow + 1, rows):
            factor = M[r][col] / M[prow][col]
            M[r] = [M[r][k] - factor * M[prow][k] for k in range(cols)]
        pivots.append(col)
        prow += 1
    return M, pivots

def show(M):
    for row in M:
        print("   [" + "  ".join(f"{str(x):>4}" for x in row) + "]")

## 1. Pivots don't have to be 1

A valid REF only needs a non-zero pivot in each non-zero row. We can optionally **divide each row by its pivot** to make every pivot a 1:

$$ \begin{bmatrix} 3 & * & * \\ 0 & 1 & * \\ 0 & 0 & -4 \end{bmatrix} \xrightarrow{\div 3,\;\div 1,\;\div(-4)} \begin{bmatrix} 1 & * & * \\ 0 & 1 & * \\ 0 & 0 & 1 \end{bmatrix} $$

Both are row echelon forms; scaling pivots to 1 makes **no mathematical difference** to rank or singularity. (This course normalises pivots to 1 by convention.)

## 2. A singular example — the staircase can skip a column

$$ \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 2 \\ 1 & 1 & 3 \end{bmatrix} $$

Subtract row 1 from rows 2 and 3, then subtract twice the new row 2 from row 3:

$$ \longrightarrow \begin{bmatrix} 1 & 1 & 1 \\ 0 & 0 & 1 \\ 0 & 0 & 0 \end{bmatrix} $$

Note the second pivot is in **column 3, not column 2** — column 2 has no pivot. Pivots need not lie on the main diagonal; they just march rightward. Two pivots → **rank 2** (singular).

In [2]:
A = [[1, 1, 1], [1, 1, 2], [1, 1, 3]]
ref, pivots = to_ref(A)
print("Row echelon form:"); show(ref)
print("\npivot columns (0-indexed):", pivots, " -> column 1 is skipped")
print("rank = number of pivots =", len(pivots))

Row echelon form:
   [   1     1     1]
   [   0     0     1]
   [   0     0     0]

pivot columns (0-indexed): [0, 2]  -> column 1 is skipped
rank = number of pivots = 2


## 3. Rank = number of pivots

Reduce each matrix to REF and count its pivots:

| Matrix | REF pivots | Rank |
|--------|-----------|------|
| $\begin{bmatrix}1&1&1\\1&2&1\\1&1&2\end{bmatrix}$ | 3 | 3 |
| $\begin{bmatrix}1&1&1\\1&1&2\\1&1&3\end{bmatrix}$ | 2 | 2 |
| $\begin{bmatrix}1&1&1\\2&2&2\\3&3&3\end{bmatrix}$ | 1 | 1 |
| $\begin{bmatrix}0&0&0\\0&0&0\\0&0&0\end{bmatrix}$ | 0 | 0 |

In [3]:
matrices = {
    "Matrix 1": [[1, 1, 1], [1, 2, 1], [1, 1, 2]],
    "Matrix 2": [[1, 1, 1], [1, 1, 2], [1, 1, 3]],
    "Matrix 3": [[1, 1, 1], [2, 2, 2], [3, 3, 3]],
    "Matrix 4": [[0, 0, 0], [0, 0, 0], [0, 0, 0]],
}
for name, M in matrices.items():
    ref, pivots = to_ref(M)
    rank = len(pivots)
    status = "non-singular" if rank == len(M) else "singular"
    print(f"{name}: pivots={rank}  rank={rank}  ({status})  numpy rank={np.linalg.matrix_rank(M)}")

Matrix 1: pivots=3  rank=3  (non-singular)  numpy rank=3
Matrix 2: pivots=2  rank=2  (singular)  numpy rank=2
Matrix 3: pivots=1  rank=1  (singular)  numpy rank=1
Matrix 4: pivots=0  rank=0  (singular)  numpy rank=0


## 4. Takeaways

- **Row echelon form (general):** zero rows at the bottom; each non-zero row starts with a **pivot**; every pivot lies strictly **right** of the one above — a descending staircase.
- **Pivots need not be 1** (or on the diagonal). Scaling them to 1 is a convention with no effect on rank or singularity; the staircase may **skip columns**.
- **Rank = number of pivots** in the REF.
- For an $n\times n$ matrix: $n$ pivots ⇔ full rank ⇔ **non-singular**; fewer pivots ⇔ **singular**.

*Companion:* [row echelon form (2×2 procedure)](./C1_W2_L2_row_echelon_form.ipynb) · [the rank of a matrix in general](./C1_W2_L2_rank_in_general.ipynb).